# LEER ARCHIVO

In [33]:
from docx import Document
from docx.oxml.ns import qn
from PIL import Image
import io
from pathlib import Path
from dataclasses import dataclass, field


@dataclass
class DocxContent:
    text: str
    images: list[Image.Image] = field(default_factory=list)


def read_docx(path: str | Path) -> DocxContent:
    """Lee un .docx y retorna el texto completo y las imágenes embebidas."""
    doc = Document(path)

    # Extraer texto párrafo a párrafo
    paragraphs = [p.text for p in doc.paragraphs]
    # También extraer texto de tablas
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                paragraphs.append(cell.text)
    text = "\n".join(p for p in paragraphs if p.strip())

    # Extraer imágenes desde las relaciones del documento
    images: list[Image.Image] = []
    for rel in doc.part.rels.values():
        if "image" in rel.reltype:
            image_data = rel.target_part.blob
            try:
                img = Image.open(io.BytesIO(image_data))
                images.append(img)
            except Exception:
                pass

    return DocxContent(text=text, images=images)


In [2]:
# Prueba
result = read_docx("pruebas/carera 26 No 50 - 47_260318_200729.docx")
print(f"Texto ({len(result.text)} chars):\n{result.text[:500]}")
print(f"\nImágenes encontradas: {len(result.images)}")


Texto (125 chars):
506
Andrea Ladino
506
Andrea Ladino
1.7+7+4.2+5.1+1.5
Curvas 1.5 m
1.7+7+4.2+5.1+1.5
Curvas 1.5 m
21 metros totales
2.100.000

Imágenes encontradas: 19


# INFERIR ELEMENTOS

In [34]:
import json
import os
import warnings
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

warnings.filterwarnings("ignore", message="Core Pydantic V1 functionality", category=UserWarning)

from langchain_core.messages import HumanMessage

load_dotenv(find_dotenv())


True

In [35]:
def load_config(data_path: str = "data.json") -> dict:
    with open(data_path, encoding="utf-8") as f:
        return json.load(f)


def build_schema_str(fields: list[dict]) -> str:
    lines = ["  {"]
    for field in fields:
        nullable = field.get("nullable", True)
        type_str = field["type"] + (" | null" if nullable else "")
        default_str = f", default={json.dumps(field['default'])}" if "default" in field else ""
        desc = field.get("description", "")
        lines.append(f'      "{field["name"]}": {type_str}{default_str}  // {desc}')
    lines.append("  }")
    return "\n".join(lines)


def load_prompt(prompt_path: str, schema: str, file_path: str, text: str) -> str:
    template = Path(prompt_path).read_text(encoding="utf-8")
    return template.format(schema=schema, file_path=file_path, text=text)


In [36]:
def get_llm(provider: str, model: str):
    match provider:
        case "anthropic":
            from langchain_anthropic import ChatAnthropic
            return ChatAnthropic(model=model, api_key=os.environ["ANTHROPIC_API_KEY"])
        case "gemini":
            from langchain_google_genai import ChatGoogleGenerativeAI
            return ChatGoogleGenerativeAI(model=model, google_api_key=os.environ["GEMINI_API_KEY"])
        case "glm":
            from langchain_openai import ChatOpenAI
            return ChatOpenAI(
                model=model,
                api_key=os.environ["ZHIPU_API_KEY"],
                base_url="https://open.bigmodel.cn/api/paas/v4/",
            )
        case "minimax":
            from langchain_anthropic import ChatAnthropic
            return ChatAnthropic(
                model=model,
                api_key=os.environ["MINIMAX_API_KEY"],
                base_url="https://api.minimaxi.chat/v1",
            )
        case _:
            raise ValueError(f"Proveedor desconocido: {provider!r}. Opciones: anthropic, gemini, glm, minimax")


In [37]:
def extract_data(
    text: str,
    file_path: str,
    provider: str = "anthropic",
    data_path: str = "data.json",
    prompt_path: str = "prompt.md",
) -> dict:
    config = load_config(data_path)
    schema_str = build_schema_str(config["fields"])
    model = config["models"][provider]

    prompt = load_prompt(prompt_path, schema=schema_str, file_path=file_path, text=text)

    llm = get_llm(provider, model)
    content = llm.invoke([HumanMessage(content=prompt)]).content

    # ChatAnthropic puede devolver lista de bloques en lugar de string
    if isinstance(content, list):
        raw = "".join(block.get("text", "") if isinstance(block, dict) else str(block) for block in content)
    else:
        raw = content

    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(raw)


In [38]:
# ── Helpers de configuración ─────────────────────────────────────────────────

def load_config(data_path: str = "data.json") -> dict:
    with open(data_path, encoding="utf-8") as f:
        return json.load(f)


def build_schema_str(fields: list[dict]) -> str:
    lines = ["  {"]
    for field in fields:
        nullable = field.get("nullable", True)
        type_str = field["type"] + (" | null" if nullable else "")
        default_str = f", default={json.dumps(field['default'])}" if "default" in field else ""
        desc = field.get("description", "")
        lines.append(f'      "{field["name"]}": {type_str}{default_str}  // {desc}')
    lines.append("  }")
    return "\n".join(lines)


def load_prompt(prompt_path: str, schema: str, file_path: str, text: str) -> str:
    template = Path(prompt_path).read_text(encoding="utf-8")
    return template.format(schema=schema, file_path=file_path, text=text)



In [39]:
# ── Fábrica de modelos LangChain ──────────────────────────────────────────────

def get_llm(provider: str, model: str):
    match provider:
        case "gemini":
            from langchain_google_genai import ChatGoogleGenerativeAI
            return ChatGoogleGenerativeAI(
                model=model, 
                google_api_key=os.environ["GEMINI_API_KEY"]
                )
        case "glm":
            from langchain_anthropic import ChatAnthropic
            return ChatAnthropic(
                model=model,
                api_key=os.environ["ZHIPU_API_KEY"],
                base_url="https://open.bigmodel.cn/api/paas/v4/",
            )
        case "minimax":
            from langchain_anthropic import ChatAnthropic
            return ChatAnthropic(
                model=model,
                api_key=os.environ["MINIMAX_API_KEY"],
                base_url="https://api.minimax.io/anthropic",
            )
        case _:
            raise ValueError(f"Proveedor desconocido: {provider!r}. Opciones: gemini, glm, minimax")



In [40]:

# ── Función principal ─────────────────────────────────────────────────────────

def extract_data(
    text: str,
    file_path: str,
    provider: str = "minimax",
    data_path: str = "data.json",
    prompt_path: str = "prompt.md",
) -> dict:
    config = load_config(data_path)
    schema_str = build_schema_str(config["fields"])
    model = config["models"][provider]

    prompt = load_prompt(prompt_path, schema=schema_str, file_path=file_path, text=text)

    llm = get_llm(provider, model)
    content = llm.invoke([HumanMessage(content=prompt)]).content

    # ChatAnthropic puede devolver lista de bloques en lugar de string
    if isinstance(content, list):
        raw = "".join(block.get("text", "") if isinstance(block, dict) else str(block) for block in content)
    else:
        raw = content

    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(raw)


In [10]:
# Prueba extract_data
fp = "pruebas/carera 26 No 50 - 47_260318_200729.docx"
content = read_docx(fp)

result = extract_data(content.text, file_path=fp, provider="minimax")
print(json.dumps(result, ensure_ascii=False, indent=2))


{
  "nombre_cliente": "Andrea Ladino",
  "direccion": "carera 26 No 50 - 47",
  "apartamento": null,
  "torre": null,
  "edificio": null,
  "costo_total": 2100000,
  "tipo_vehiculo": "Tesla"
}


# CONFIRMAR ELEMENTOS

In [41]:
import ipywidgets as w
from IPython.display import display as ipy_display


class DocumentEditor:
    """Muestra el documento y un formulario editable con los datos extraídos."""

    def __init__(self, doc: DocxContent, data: dict):
        self._field_rows: list[tuple[w.Text, w.Text, w.HBox]] = []
        self._fields_box = w.VBox()
        self._build(doc, data)

    # ── Construcción ──────────────────────────────────────────────────────────

    def _build(self, doc: DocxContent, data: dict):
        txt = w.Textarea(
            value=doc.text,
            layout=w.Layout(width="100%", height="400px"),
            disabled=True,
        )
        img_widgets = []
        for img in doc.images[:12]:
            buf = io.BytesIO()
            thumb = img.copy()
            thumb.thumbnail((110, 110))
            thumb.save(buf, format="PNG")
            img_widgets.append(w.Image(value=buf.getvalue(), width=110, height=110))

        left = w.VBox(
            [
                w.HTML("<b>Documento</b>"),
                txt,
                w.GridBox(
                    img_widgets,
                    layout=w.Layout(grid_template_columns="repeat(4, 114px)", gap="4px"),
                ),
            ],
            layout=w.Layout(width="50%", padding="0 12px 0 0"),
        )

        for key, val in data.items():
            self._add_row(key, "" if val is None else str(val))

        btn_add = w.Button(description="+ Agregar campo", button_style="info", icon="plus")
        btn_add.on_click(lambda _: self._add_row("", ""))

        right = w.VBox(
            [w.HTML("<b>Datos extraídos</b>"), self._fields_box, btn_add],
            layout=w.Layout(width="50%"),
        )

        self._root = w.HBox([left, right], layout=w.Layout(width="100%"))

    def _add_row(self, key: str, value: str):
        key_w = w.Text(value=key, placeholder="campo", layout=w.Layout(width="160px"))
        val_w = w.Text(value=value, placeholder="valor", layout=w.Layout(width="260px"))
        del_btn = w.Button(
            icon="trash", button_style="danger",
            layout=w.Layout(width="36px", height="32px"),
        )
        row = w.HBox([key_w, val_w, del_btn])

        def on_delete(_):
            self._field_rows[:] = [(k, v, r) for k, v, r in self._field_rows if r is not row]
            self._fields_box.children = [r for _, _, r in self._field_rows]

        del_btn.on_click(on_delete)
        self._field_rows.append((key_w, val_w, row))
        self._fields_box.children = (*self._fields_box.children, row)

    @property
    def values(self) -> dict:
        """Retorna el dict con los valores actuales del formulario."""
        result = {}
        for key_w, val_w, _ in self._field_rows:
            k = key_w.value.strip()
            if k:
                v = val_w.value.strip()
                result[k] = None if v == "" else v
        return result

    def display(self):
        ipy_display(self._root)


In [42]:
# Uso
fp = "pruebas/carera 26 No 50 - 47_260318_200729.docx"
doc = read_docx(fp)
extracted = extract_data(doc.text, file_path=fp)


In [43]:
editor = DocumentEditor(doc, extracted)
editor.display()

In [46]:
print(extracted)
extracted["apartamento"] = "503"

{'nombre': 'Andrea Ladino', 'is_male': False, 'direccion': 'carera 26 No 50 - 47', 'apartamento': '503', 'nombre_edificio': None, 'costo_total': 2100000, 'ciudad_departamento': 'Bogotá D.C.', 'tipo_vehiculo': 'Tesla'}


# GENERAR DOCUMENTO

In [77]:
from docxtpl import DocxTemplate, RichText
from datetime import date
import re

_MESES = [
    "enero", "febrero", "marzo", "abril", "mayo", "junio",
    "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre",
]

_IF_BLOCK = re.compile(r'\{%-?\s*if\s+\w+\s*-?%\}.*\{%-?\s*endif\s*-?%\}', re.DOTALL)


def _make_page_break() -> RichText:
    rt = RichText()
    rt.add("", break_type="page")
    return rt


def _conditional_para_indices(template_path: str) -> set[int]:
    """Índices de párrafos del template que son bloques {% if %}...{% endif %}."""
    doc = Document(template_path)
    return {i for i, p in enumerate(doc.paragraphs) if _IF_BLOCK.search(p.text)}


def _remove_null_paragraphs(output_path: Path, indices: set[int]):
    """Elimina solo los párrafos que vinieron de un {% if %} y quedaron vacíos."""
    doc = Document(output_path)
    paras = doc.paragraphs
    for i in sorted(indices, reverse=True):
        if i < len(paras) and paras[i].text.strip() == "":
            paras[i]._element.getparent().remove(paras[i]._element)
    doc.save(output_path)


def fill_template(
    data: dict,
    template_path: str = "pruebas/template_base.docx",
    output_dir: str = "pruebas",
    fotos: list[dict] | None = None,
) -> Path:
    """Rellena template_base.docx con los datos extraídos y guarda el resultado."""
    cond_indices = _conditional_para_indices(template_path)

    tpl = DocxTemplate(template_path)

    hoy = date.today()
    fecha = f"{hoy.day} de {_MESES[hoy.month - 1]} de {hoy.year}"

    ciudad_dep = data.get("ciudad_departtamento") or data.get("ciudad_departamento") or "Bogotá D.C."
    ciudad = ciudad_dep.split(",")[0].strip()

    context = {
        "NOMBRE":              (data.get("nombre") or "").upper(),
        "señor_señora":        "señor" if data.get("is_male", True) else "señora",
        "direccion":           data.get("direccion", ""),
        "apartamento":         data.get("apartamento"),
        "nombre_edificio":     data.get("nombre_edificio"),
        "ciudad_departamento": ciudad_dep,
        "ciudad":              ciudad,
        "fecha":               fecha,
        "tipo_vehiculo":       data.get("tipo_vehiculo") or "Tesla",
        "fotos":               fotos or [],
        "new_page":           RichText('\f'),
    }

    tpl.render(context)

    direccion_slug = re.sub(r'[^\w\s-]', '', data.get("direccion", "documento")).strip().replace(" ", "_")
    timestamp = hoy.strftime("%Y%m%d")
    output_path = Path(output_dir) / f"{direccion_slug}_{timestamp}.docx"
    tpl.save(output_path)

    _remove_null_paragraphs(output_path, cond_indices)
    return output_path


In [78]:
extracted

{'nombre': 'Andrea Ladino',
 'is_male': False,
 'direccion': 'carera 26 No 50 - 47',
 'apartamento': '503',
 'nombre_edificio': None,
 'costo_total': 2100000,
 'ciudad_departamento': 'Bogotá D.C.',
 'tipo_vehiculo': 'Tesla'}

In [79]:
# Uso
output = fill_template(extracted)
print(f"Guardado en: {output}")


Guardado en: pruebas/carera_26_No_50_-_47_20260401.docx
